# SPNER: Kaggle GPU validation gates
Clone one exact GitHub commit. Keep this notebook private. Run smoke, then tiny overfit, then the monolingual baseline. The confidential FilNER manual is never an input to this notebook.
GPU execution has not yet been validated for this starter. Save a Kaggle version with outputs to retain checkpoints and logs.

In [ ]:
REPO_URL = "https://github.com/Ka1iZan/-Special-Problem-.git"
GIT_COMMIT = ""  # exact 40-character commit SHA from the tested local code
PRIVATE_REPO = False
GITHUB_SECRET_NAME = "SPNER_GITHUB_TOKEN"  # Kaggle secret; read-only repository access
MODE = "smoke"  # then tiny; then baseline, after each prior gate succeeds
RUN_NAME = "tag_smoke_seed42"  # choose a fresh name per run


In [ ]:
import os, sys, re, subprocess, base64
from pathlib import Path
assert sys.version_info >= (3, 11), "Kaggle training dependencies require Python 3.11+"
assert REPO_URL.startswith("https://github.com/") and "@" not in REPO_URL, "Set a credential-free GitHub URL"
assert re.fullmatch(r"[0-9a-fA-F]{40}", GIT_COMMIT), "Set the exact tested commit"
repo = Path("/kaggle/working/spner")
git_env = os.environ.copy()
if PRIVATE_REPO:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(GITHUB_SECRET_NAME)
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0="Authorization: Basic " + base64.b64encode(("x-access-token:" + token).encode()).decode())
    del token
if not repo.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(repo)], env=git_env, check=True)
os.chdir(repo)
assert subprocess.check_output(["git", "remote", "get-url", "origin"], text=True).strip() == REPO_URL
subprocess.run(["git", "fetch", "origin", GIT_COMMIT], env=git_env, check=True)
subprocess.run(["git", "checkout", "--detach", GIT_COMMIT], check=True)
del git_env
print("Code commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-kaggle.txt"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_core*.py", "-v"], check=True)
subprocess.run([sys.executable, "scripts/doctor.py"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "tests/test_prototypes.py", "tests/test_harmonization.py", "-q"], check=True)
import torch
assert torch.cuda.is_available(), "Enable GPU in Kaggle notebook settings"


In [ ]:
if not Path("data/raw/public_v1/manifest.json").exists():
    subprocess.run([sys.executable, "scripts/fetch_public_data.py"], check=True)
if not Path("data/processed/tlunified_v1/manifest.json").exists():
    subprocess.run([sys.executable, "scripts/prepare_tlunified.py"], check=True)
# This first gate uses the public TLUnified-NER release only.
# Other corpora need their documented format/ontology decisions before training.


In [ ]:
assert MODE in {"smoke", "tiny", "baseline"}
assert re.fullmatch(r"[a-zA-Z0-9_-]+", RUN_NAME)
run_dir = Path("/kaggle/working/spner_runs") / RUN_NAME
subprocess.run([sys.executable, "scripts/train_baseline.py", "--data", "data/processed/tlunified_v1",
                "--config", "configs/baseline.json", "--mode", MODE, "--out", str(run_dir)], check=True)
print("Keep these run outputs:", run_dir)
# Test evaluation is intentionally absent until model-selection rules are frozen.
